# Этап 3. Предобработка данных: Titanic

Здесь данные готовятся для будущей модели. Главное правило: сначала делим данные на обучающую и тестовую части, а затем все настройки делаем только по обучающей части. Так тестовые данные остаются честной проверкой.


## 1. Загрузка данных


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, OneHotEncoder
from IPython.display import display

# В репозитории CSV находится в папке data.
# В Google Colab, если файла нет, появится окно для его загрузки.
DATA_PATH = Path('data/titanic_train.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('titanic_train.csv')
if not DATA_PATH.exists():
    from google.colab import files
    print('Выберите файл titanic_train.csv')
    uploaded = files.upload()
    DATA_PATH = Path(next(iter(uploaded)))

df = pd.read_csv(DATA_PATH)
print(f'Загружено строк: {len(df)}')
df.head()


## 2. Удаление дубликатов и логических ошибок

Это безопасная очистка: она не использует информацию о выживании для заполнения данных. Пропуски специально пока не заполняю — сделаю это только после разделения на train и test.


In [ ]:
df_clean = df.drop_duplicates().copy()
valid = (
    df_clean['Survived'].isin([0, 1])
    & df_clean['Pclass'].isin([1, 2, 3])
    & (df_clean['Age'].isna() | df_clean['Age'].between(0, 100))
    & (df_clean['Fare'] >= 0)
)
df_clean = df_clean.loc[valid].copy()

print(f'Строк после базовой очистки: {len(df_clean)}')
print('Пропуски до заполнения:')
display(df_clean.isna().sum()[df_clean.isna().sum() > 0].to_frame('Количество'))


## 3. Деление на train и test — 80/20

`train` нужен, чтобы настроить обработку. `test` оставляем как отдельную проверку. `stratify=y` сохраняет примерно одинаковую долю выживших в обеих частях.


In [ ]:
X = df_clean.drop(columns='Survived')
y = df_clean['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Обучающая часть: {len(X_train)} строк ({len(X_train) / len(X):.0%})')
print(f'Тестовая часть: {len(X_test)} строк ({len(X_test) / len(X):.0%})')


## 4. Масштабирование признака `Fare`

`Fare` — стоимость билета. В этом столбце есть очень дорогие билеты, поэтому сравниваю четыре метода из шаблона. Каждый метод настраивается только на `train`.


In [ ]:
def compare_distribution(before, after, method_name):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    axes[0].hist(before, bins=25, color='#4c78a8', edgecolor='white')
    axes[0].set_title('До: Fare')
    axes[1].hist(after, bins=25, color='#72b7b2', edgecolor='white')
    axes[1].set_title(f'После: {method_name}')
    plt.tight_layout()
    plt.show()

fare_train = X_train[['Fare']]
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'MaxAbsScaler': MaxAbsScaler(),
    'RobustScaler': RobustScaler(),
}

scaled_examples = pd.DataFrame({'До': fare_train['Fare'].head(8).values})
for name, scaler in scalers.items():
    after = scaler.fit_transform(fare_train).ravel()
    scaled_examples[name] = after[:8].round(2)
    compare_distribution(fare_train['Fare'], after, name)

display(scaled_examples)


## 5. Выбор метода

Для `Fare` выбираю `RobustScaler`: он опирается на медиану и межквартильный размах, поэтому большие значения влияют на него меньше. Перед ним применяю `log1p`, чтобы сделать распределение цены менее перекошенным.


## 6. Обработка остальных признаков

- `PassengerId` — просто номер, его убираю.
- `Name` и `Ticket` почти уникальны у каждого пассажира, поэтому в простой версии их тоже убираю.
- Из `Cabin` создаю `HasCabin`: известна каюта или нет, затем исходный столбец убираю.
- Числовой `Age` заполняю медианой из train.
- Категориальный `Embarked` заполняю самым частым значением из train.
- `Pclass`, `Sex`, `Embarked` и `HasCabin` превращаю в отдельные столбцы 0/1 через `OneHotEncoder`.


In [ ]:
def simple_features(data):
    result = data.copy()
    result['HasCabin'] = result['Cabin'].notna().astype(str)
    return result.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

train_ready = simple_features(X_train)
test_ready = simple_features(X_test)

# Числовые признаки: медианы считаем только по train.
numeric_columns = ['Age', 'SibSp', 'Parch', 'Fare']
for column in numeric_columns:
    fill_value = train_ready[column].median()
    train_ready[column] = train_ready[column].fillna(fill_value)
    test_ready[column] = test_ready[column].fillna(fill_value)

# Fare: сначала уменьшаем перекос, затем применяем выбранный RobustScaler.
fare_scaler = RobustScaler()
train_ready['Fare'] = fare_scaler.fit_transform(np.log1p(train_ready[['Fare']])).ravel()
test_ready['Fare'] = fare_scaler.transform(np.log1p(test_ready[['Fare']])).ravel()

# Категории: пропуск берём из train, новые категории в test не вызывают ошибку.
categorical_columns = ['Pclass', 'Sex', 'Embarked', 'HasCabin']
for column in categorical_columns:
    fill_value = train_ready[column].mode()[0]
    train_ready[column] = train_ready[column].fillna(fill_value).astype(str)
    test_ready[column] = test_ready[column].fillna(fill_value).astype(str)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_categorical = encoder.fit_transform(train_ready[categorical_columns])
test_categorical = encoder.transform(test_ready[categorical_columns])

encoded_names = encoder.get_feature_names_out(categorical_columns)
train_final = pd.concat([
    train_ready[numeric_columns].reset_index(drop=True),
    pd.DataFrame(train_categorical, columns=encoded_names),
], axis=1)
test_final = pd.concat([
    test_ready[numeric_columns].reset_index(drop=True),
    pd.DataFrame(test_categorical, columns=encoded_names),
], axis=1)

print('Размер train после обработки:', train_final.shape)
print('Размер test после обработки:', test_final.shape)
print('Пропусков в train:', train_final.isna().sum().sum())
print('Пропусков в test:', test_final.isna().sum().sum())
display(train_final.head())


## 7. Вывод

Данные разделены в пропорции 80/20 и подготовлены для обучения модели. В итоговых таблицах нет пропусков, все категориальные признаки переведены в числа, а `Fare` масштабирован методом, который устойчивее к очень дорогим билетам.
